# Adding a 6th critic: Reachability

Crucible's five default critics cover visual, kinematic, task,
strategy, and safety. This notebook walks through adding a sixth
critic — **Reachability** — that scores whether the demonstrated
trajectory uses arm geometry that will generalize to a different
physical robot (e.g. ALOHA → bimanual UR5).

The pattern is identical for any critic axis you want to add: write
a prompt, declare the verdict vocabulary, the rest is one tuple
registration in `src/critics.py`.

Total runtime: ~5 minutes. ~$0.05 in API spend.

## Step 1 — The new critic prompt

We'll keep this notebook self-contained: the prompt is defined as a
string here rather than written to `src/prompts/critic_reachability.txt`.
If you want to upstream this critic, copy the prompt into a file and
follow CONTRIBUTING.md.

In [ ]:
REACHABILITY_PROMPT = '''
You are a Reachability Critic for robot teleoperation demonstrations.
You will be given:
- A task description.
- A few reference frames spanning the episode.
- A telemetry digest of joint positions and velocities.

Score 0-10 whether the demonstrated trajectory uses arm geometry that
will generalize to a different bimanual robot platform with different
link lengths or joint limits. Specifically:

- Does the trajectory hug the workspace edges in a way that exploits
  this robot's specific reach limits (won't generalize)?
- Does the operator use shoulder-rotation moves that would be
  awkward on a robot with different joint stops (won't generalize)?
- Does the approach work in the middle of the workspace where most
  bimanual robots have ample reach (will generalize)?
- Does the trajectory respect a sphere of typical bimanual reach
  rather than relying on this robot's extremes (will generalize)?

Score 10 = transfers cleanly to any reasonable bimanual platform.
Score 5 = partial generalization; some moves will not transfer.
Score 0 = trajectory only works on this robot's specific geometry.

Output strict JSON only with fields: score, verdict, rationale, evidence.
verdict is one of: GENERALIZES, MOSTLY_GENERALIZES, ROBOT_SPECIFIC, NONTRANSFERABLE.
'''.strip()

REACHABILITY_VERDICT_VOCAB = ['GENERALIZES', 'MOSTLY_GENERALIZES', 'ROBOT_SPECIFIC', 'NONTRANSFERABLE']

REACHABILITY_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'required': ['score', 'verdict', 'rationale', 'evidence'],
    'properties': {
        'score': {'type': 'number', 'minimum': 0, 'maximum': 10},
        'verdict': {'type': 'string', 'enum': REACHABILITY_VERDICT_VOCAB},
        'rationale': {'type': 'string', 'minLength': 1, 'maxLength': 1200},
        'evidence': {
            'type': 'array',
            'maxItems': 8,
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'required': ['timestamp', 'observation'],
                'properties': {
                    'timestamp': {'type': 'string'},
                    'observation': {'type': 'string', 'maxLength': 240},
                },
            },
        },
    },
}

## Step 2 — Pull one episode and run the new critic

In [ ]:
import os, sys, asyncio, json
sys.path.insert(0, '..')

from openai import AsyncOpenAI
from src.config import CrucibleConfig
from src.lerobot_io import stream_episodes
from src.critics import build_user_message, _chat_with_retries, _extract_json_loose

cfg = CrucibleConfig()

bundle = next(stream_episodes('lerobot/aloha_static_cups_open', n=1, frames_per_episode=8))
client = AsyncOpenAI(base_url=cfg.vlm_endpoint, api_key=cfg.vlm_api_key)

# Build the user message using Crucible's helper. We pass `name='strategy'`
# only to satisfy the helper's frame-slicing logic — the system prompt is
# what determines the critic's behavior.
user_content = build_user_message('strategy', bundle, cfg)

async def run_reachability():
    raw = await _chat_with_retries(
        client, cfg,
        system=REACHABILITY_PROMPT,
        user_content=user_content,
        max_tokens=cfg.critic_max_tokens,
        temperature=cfg.critic_temperature,
        schema=REACHABILITY_SCHEMA,
        schema_name='critic_reachability_verdict',
    )
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return _extract_json_loose(raw)

result = asyncio.run(run_reachability())
print(json.dumps(result, indent=2, default=str))

## Step 3 — Upstreaming this critic

If you want to make Reachability a permanent part of Crucible:

1. Save `REACHABILITY_PROMPT` to `src/prompts/critic_reachability.txt`
   (without the surrounding triple quotes).
2. In `src/critics.py`, add `'reachability'` to `CRITIC_NAMES`.
3. Add `'reachability': REACHABILITY_VERDICT_VOCAB` to `CRITIC_VERDICT_VOCAB`.
4. Add a frame-slicing branch in `_frames_to_send` if you want a
   different sample (otherwise it'll get the default full-episode
   set).
5. Add `'reachability'` to the `storage_keys` list at the bottom
   of `run_all_critics`.
6. In `src/aggregator.py`, add a `'reachability': 1.0` entry to
   `CRITIC_WEIGHTS` (or weight it higher / lower as fits the rubric).
7. Add a test in `tests/test_aggregator.py` covering the new critic
   in the verdict-rule scenarios.

Send a PR with these changes plus rationale for why this axis matters
for the curation pipeline.